# Training Spike - SEA-LION v4.5 E2B vs v3 9B

**Purpose: de-risk the stack before Phase 3 builds 800-1500 examples.**

Answers four questions:
1. Does each model load in 4-bit on a free T4?
2. Does Unsloth take LoRA steps on it?
3. How much VRAM, how long per step?
4. Does it emit coherent, grounded **Burmese** - is ~2B enough, or is 9B needed?

Throwaway 18-example seed set. NOT the Phase 3 dataset.

**Runtime -> Change runtime type -> T4 GPU** before running.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install unsloth
!pip -q install --no-deps --upgrade peft accelerate bitsandbytes

## Seed data

The repo is **private**, so the raw GitHub URL will 404. The cell below tries it anyway (in case you make the repo public) and otherwise prompts you to upload `data/samples/spike_seed.jsonl` from your machine.

In [ ]:
import json, urllib.request, time, gc, torch

# The repo is PRIVATE, so raw.githubusercontent.com will 404 without a token.
# Try the URL anyway (works if you make the repo public), else upload the file.
URL = ("https://raw.githubusercontent.com/Thant9330/EPS-Burmese-Assistant/"
       "master/data/samples/spike_seed.jsonl")

raw = None
try:
    raw = urllib.request.urlopen(URL, timeout=20).read().decode()
    print("loaded from GitHub raw URL")
except Exception as e:
    print("raw URL unavailable (", type(e).__name__, ") - falling back to upload")
    print("Pick data/samples/spike_seed.jsonl from your machine:")
    from google.colab import files
    up = files.upload()
    raw = list(up.values())[0].decode("utf-8")

rows = [json.loads(l) for l in raw.splitlines() if l.strip()]
print(len(rows), "examples |",
      "grounded=", sum(r["kind"] == "grounded" for r in rows),
      "refusal=", sum(r["kind"] == "refusal" for r in rows))

# Hold two out (one grounded, one refusal) so BEFORE/AFTER is on unseen questions.
HELD = [rows[2], rows[-1]]
TRAIN = [r for r in rows if r not in HELD]
print("train=", len(TRAIN), " held_out=", len(HELD))
print("held-out Q1:", HELD[0]["question"], "|", HELD[0]["kind"])
print("held-out Q2:", HELD[1]["question"], "|", HELD[1]["kind"])

## The experiment

Same data, same LoRA config, same steps - only the base model changes.
Each model is wrapped in try/except so one failure still leaves the other's numbers.

In [ ]:
MODELS = [
    ("aisingapore/Gemma-SEA-LION-v4.5-E2B-IT", "~2B eff, gemma4, 1.85x Burmese"),
    ("aisingapore/Gemma-SEA-LION-v3-9B-IT",    "9B, gemma2, 3.32x Burmese"),
]
MAX_SEQ, STEPS = 2048, 30
results = {}

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset


def prompt_of(r):
    return (r["system"] + "\n\n### Context\n" + r["context"]
            + "\n\n### Question\n" + r["question"])


def build_text(tok, r):
    msgs = [{"role": "user", "content": prompt_of(r)},
            {"role": "assistant", "content": r["answer"]}]
    return tok.apply_chat_template(msgs, tokenize=False)


def generate(model, tok, r, n=220):
    FastLanguageModel.for_inference(model)
    ids = tok.apply_chat_template([{"role": "user", "content": prompt_of(r)}],
                                  tokenize=True, add_generation_prompt=True,
                                  return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=n, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)


def run(name, note):
    print("=" * 78)
    print(name, "|", note)
    print("=" * 78, flush=True)
    rec = {"note": note}

    t0 = time.time()
    model, tok = FastLanguageModel.from_pretrained(
        model_name=name, max_seq_length=MAX_SEQ, dtype=None, load_in_4bit=True)
    rec["load_s"] = round(time.time() - t0, 1)
    rec["vram_load_GB"] = round(torch.cuda.memory_allocated() / 1e9, 2)
    print("loaded in", rec["load_s"], "s | VRAM", rec["vram_load_GB"], "GB")

    rec["before"] = [generate(model, tok, h) for h in HELD]
    print("--- BEFORE training ---")
    for i, b in enumerate(rec["before"]):
        print("[", i, "]", b[:400])

    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=0)

    ds = Dataset.from_list([{"text": build_text(tok, r)} for r in TRAIN])
    trainer = SFTTrainer(
        model=model, tokenizer=tok, train_dataset=ds,
        args=SFTConfig(per_device_train_batch_size=1, gradient_accumulation_steps=4,
                       warmup_steps=5, max_steps=STEPS, learning_rate=2e-4,
                       logging_steps=5, optim="adamw_8bit", weight_decay=0.01,
                       lr_scheduler_type="linear", seed=0, output_dir="out",
                       report_to="none", max_length=MAX_SEQ,
                       dataset_text_field="text"))

    t1 = time.time()
    stats = trainer.train()
    rec["train_s"] = round(time.time() - t1, 1)
    rec["s_per_step"] = round(rec["train_s"] / STEPS, 2)
    rec["final_loss"] = round(stats.training_loss, 4)
    rec["peak_vram_GB"] = round(torch.cuda.max_memory_reserved() / 1e9, 2)
    print("trained", STEPS, "steps in", rec["train_s"], "s |",
          rec["s_per_step"], "s/step | loss", rec["final_loss"],
          "| peak", rec["peak_vram_GB"], "GB")

    rec["after"] = [generate(model, tok, h) for h in HELD]
    print("--- AFTER training ---")
    for i, a in enumerate(rec["after"]):
        print("[", i, "]", a[:400])

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    return rec


for _name, _note in MODELS:
    try:
        results[_name] = run(_name, _note)
    except Exception as e:
        results[_name] = {"note": _note, "ERROR": type(e).__name__ + ": " + str(e)}
        print("!!! FAILED:", type(e).__name__, e)
        gc.collect()
        torch.cuda.empty_cache()

## Comparison - this is what decides the base model

In [ ]:
hdr = "%-42s %6s %7s %8s %7s" % ("model", "load", "s/step", "peakGB", "loss")
print(hdr)
print("-" * len(hdr))
for k, v in results.items():
    short = k.split("/")[-1]
    if "ERROR" in v:
        print("%-42s  FAILED: %s" % (short, v["ERROR"][:30]))
    else:
        print("%-42s %6s %7s %8s %7s" % (short, v["load_s"], v["s_per_step"],
                                         v["peak_vram_GB"], v["final_loss"]))

print("")
print("BURMESE QUALITY - judge yourself; no metric replaces a native reader:")
for k, v in results.items():
    if "ERROR" in v:
        continue
    print("")
    print("#" * 78)
    print("#", k)
    print("#" * 78)
    for i in range(len(HELD)):
        print("")
        print("--- held-out Q%d (%s): %s" % (i + 1, HELD[i]["kind"], HELD[i]["question"]))
        print("  BEFORE:", v["before"][i][:500])
        print("  AFTER :", v["after"][i][:500])

print("")
print("JSON to paste back into the repo:")
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk not in ("before", "after")}
                  for k, v in results.items()}, indent=2))

## What to look for

**Stack works** - both load in 4-bit, take steps, peak VRAM comfortably under 15GB.

**Burmese quality** - the deciding question. Compare BEFORE vs AFTER for both:
- Is the Burmese grammatical and natural?
- Are Korean terms (고용센터, 외국인등록) preserved rather than mangled?
- Does it stay grounded in the context instead of inventing rules?
- Does the held-out refusal example actually refuse?

**If E2B's Burmese is close to 9B's** - take E2B: 1.85x tokenizer, ~4x faster iteration,
and the same tokenizer as v4-27B, so scaling up later needs no re-chunk.

**If 9B is clearly better** - the 3.32x token cost is the price of quality. Better to learn
that now than after building 1500 examples.

Paste the JSON block back to Claude to have it written into the repo.